# Pothole & Speed Breaker Detection (Python)

This notebook has **two complementary approaches**:

| Part | Input | Method | Runs without extra data? |
|------|-------|--------|--------------------------|
| **1. Sensor-based** | Accelerometer (phone / IMU) z-axis | Sliding window + features + Random Forest | Yes (synthetic data included) |
| **2. Vision-based** | Dashcam images / video | YOLOv8 object detection | Needs a labelled dataset for training |

**Why both?** Accelerometers tell you *that* the vehicle hit something (and work at night), cameras tell you *what* is ahead *before* you hit it. Combining them is the most robust approach.

Key idea for Part 1: a **pothole** produces a short, sharp drop-and-rebound spike (0.2-0.4 s), while a **speed breaker** produces a longer, smoother bump (0.8-1.5 s).

In [ ]:
%pip install -q numpy pandas scipy scikit-learn matplotlib opencv-python ultralytics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

plt.rcParams['figure.figsize'] = (12, 4)

---
# Part 1 - Accelerometer-based detection

## 1.1 Generate synthetic road data

We simulate a vertical-axis accelerometer (`az`, m/s^2) at 50 Hz with road noise plus randomly placed potholes and speed breakers. **To use real data**, skip to section 1.6 and load your own CSV instead.

In [ ]:
FS = 50  # sampling rate (Hz)
LABELS = {0: 'normal', 1: 'pothole', 2: 'speed_breaker'}

def make_signal(duration_s=600, seed=0):
    rng = np.random.default_rng(seed)
    n = int(duration_s * FS)
    t = np.arange(n) / FS
    # base: gravity + road roughness + slow suspension sway
    z = 9.81 + rng.normal(0, 0.25, n) + 0.3 * np.sin(2 * np.pi * 0.5 * t)
    y = np.zeros(n, dtype=int)
    pos = 5 * FS
    while pos < n - 4 * FS:
        kind = int(rng.choice([1, 2]))
        if kind == 1:   # pothole: sharp drop + rebound, short
            dur = int(rng.uniform(0.2, 0.4) * FS)
            amp = rng.uniform(2.5, 5.0)
            k = np.arange(dur)
            pulse = -amp * np.sin(2 * np.pi * k / dur) * np.exp(-2 * k / dur)
        else:           # speed breaker: smooth, longer bump
            dur = int(rng.uniform(0.8, 1.5) * FS)
            amp = rng.uniform(1.5, 3.0)
            k = np.arange(dur)
            pulse = amp * np.sin(np.pi * k / dur) - 0.5 * amp * np.sin(2 * np.pi * k / dur)
        z[pos:pos + dur] += pulse
        y[pos:pos + dur] = kind
        pos += dur + int(rng.uniform(6, 15) * FS)
    return pd.DataFrame({'t': t, 'az': z, 'label': y})

train_df = make_signal(600, seed=0)
test_df = make_signal(300, seed=1)   # separate drive -> no leakage
print(train_df.shape, test_df.shape)
train_df.head()

In [ ]:
# Look at the first 60 seconds with ground-truth events shaded
def plot_signal(df, title, t0=0, t1=60, pred_events=None):
    d = df[(df.t >= t0) & (df.t < t1)]
    fig, ax = plt.subplots()
    ax.plot(d.t, d.az, lw=0.8, color='steelblue')
    colors = {1: 'red', 2: 'orange'}
    if 'label' in d and pred_events is None:
        for k, c in colors.items():
            m = (d.label == k).values
            ax.fill_between(d.t, d.az.min(), d.az.max(), where=m, color=c, alpha=0.3, label=LABELS[k])
    if pred_events is not None:
        for (s, e, k) in pred_events:
            if s < t1 and e > t0:
                ax.axvspan(s, e, color=colors[k], alpha=0.3)
        ax.plot([], [], color='red', alpha=0.3, lw=8, label='pothole (pred)')
        ax.plot([], [], color='orange', alpha=0.3, lw=8, label='speed_breaker (pred)')
    ax.set(title=title, xlabel='time (s)', ylabel='az (m/s^2)')
    ax.legend(loc='upper right')
    plt.show()

plot_signal(train_df, 'Synthetic accelerometer data with ground truth')

## 1.2 Windowing and feature extraction

We slice the signal into 1.5 s windows (50% overlap-ish) and compute shape features. Potholes have **high peak jerk and kurtosis**; speed breakers have **lower jerk but a longer period above threshold**.

In [ ]:
WIN, STEP = int(1.5 * FS), int(0.5 * FS)
FEATURES = ['std', 'ptp', 'min', 'max', 'peak_jerk', 'rms', 'skew', 'kurtosis', 'n_above_1ms2']

def window_features(w):
    w = w - w.mean()                      # remove gravity / offset
    jerk = np.diff(w) * FS                # rate of change of acceleration
    return [
        w.std(), np.ptp(w), w.min(), w.max(),
        np.abs(jerk).max(), np.sqrt((w ** 2).mean()),
        skew(w), kurtosis(w), int((np.abs(w) > 1.0).sum()),
    ]

def build_windows(df):
    az, lab = df['az'].values, df['label'].values
    X, y, starts = [], [], []
    for s in range(0, len(az) - WIN, STEP):
        X.append(window_features(az[s:s + WIN]))
        counts = np.bincount(lab[s:s + WIN], minlength=3)
        # label a window as an event if >= 10 samples (0.2 s) overlap it
        y.append(int(np.argmax(counts[1:]) + 1) if counts[1:].max() >= 10 else 0)
        starts.append(s)
    return pd.DataFrame(X, columns=FEATURES), np.array(y), np.array(starts)

X_train, y_train, _ = build_windows(train_df)
X_test, y_test, starts_test = build_windows(test_df)
print('train windows:', X_train.shape, ' class counts:', np.bincount(y_train))
print('test windows :', X_test.shape, ' class counts:', np.bincount(y_test))

## 1.3 Train a Random Forest classifier

In [ ]:
clf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

pred = clf.predict(X_test)
print(classification_report(y_test, pred, target_names=list(LABELS.values()), zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=list(LABELS.values()), cmap='Blues')
plt.title('Confusion matrix (test drive)')
plt.show()

imp = pd.Series(clf.feature_importances_, index=FEATURES).sort_values()
imp.plot.barh(title='Feature importance')
plt.show()

## 1.4 Turn window predictions into events

Consecutive windows with the same predicted class are merged into one event with a start/end time.

In [ ]:
def windows_to_events(pred, starts):
    events, cur = [], None
    for p, s in zip(pred, starts):
        if p != 0 and cur and cur[2] == p and s / FS <= cur[1]:
            cur[1] = (s + WIN) / FS           # extend
        elif p != 0:
            if cur: events.append(tuple(cur))
            cur = [s / FS, (s + WIN) / FS, int(p)]
        else:
            if cur: events.append(tuple(cur)); cur = None
    if cur: events.append(tuple(cur))
    return events

events = windows_to_events(pred, starts_test)
ev_df = pd.DataFrame(events, columns=['start_s', 'end_s', 'class_id'])
ev_df['type'] = ev_df['class_id'].map(LABELS)
print(ev_df['type'].value_counts().to_string())
ev_df.head(10)

In [ ]:
plot_signal(test_df, 'Predicted events on an unseen drive', 0, 60, pred_events=events)

## 1.5 Simple rule-based baseline (no ML)

Useful as a sanity check or for low-power devices: classify by peak magnitude and event duration.

In [ ]:
def rule_based_detect(az, thr=1.2, min_gap_s=1.0):
    dev = az - np.median(az)
    idx = np.where(np.abs(dev) > thr)[0]
    if len(idx) == 0:
        return []
    groups = np.split(idx, np.where(np.diff(idx) > min_gap_s * FS)[0] + 1)
    out = []
    for g in groups:
        dur = (g[-1] - g[0] + 1) / FS
        kind = 1 if dur < 0.6 else 2          # short & sharp = pothole, long = speed breaker
        out.append((g[0] / FS, g[-1] / FS, kind))
    return out

rb = rule_based_detect(test_df['az'].values)
print(len(rb), 'events found by rules;', int((np.diff((test_df.label.values > 0).astype(int)) == 1).sum()), 'true events')

## 1.6 Use your own real data

Record with a phone app such as *Phyphox*, *Sensor Logger* or *AndroidSensor*, mount the phone flat in the vehicle, and export a CSV with a `time` column (seconds) and a vertical acceleration column. Resample to 50 Hz, then reuse the trained model:

In [ ]:
def detect_from_csv(path, time_col='time', az_col='az'):
    raw = pd.read_csv(path)
    t = raw[time_col].values - raw[time_col].values[0]
    grid = np.arange(0, t[-1], 1 / FS)
    az = np.interp(grid, t, raw[az_col].values)          # resample to FS
    df = pd.DataFrame({'t': grid, 'az': az, 'label': np.zeros(len(grid), dtype=int)})
    X, _, starts = build_windows(df)
    ev = windows_to_events(clf.predict(X), starts)
    out = pd.DataFrame(ev, columns=['start_s', 'end_s', 'class_id'])
    out['type'] = out['class_id'].map(LABELS)
    return df, out

# df, events = detect_from_csv('my_drive.csv', time_col='time', az_col='az')
# events

**Tips for real data:** log GPS alongside the accelerometer so each event gets a latitude/longitude (for a pothole map); events depend on vehicle speed, so normalise by speed or train separate models per speed band; and train with real labelled drives, since synthetic results will look better than real-world ones.

---
# Part 2 - Vision-based detection with YOLOv8

## 2.1 Get a dataset

Download a labelled pothole dataset in **YOLOv8 format** (free options: Roboflow Universe -> search *pothole detection*, or Kaggle -> *Pothole Detection Dataset*). For speed breakers, search *speed bump detection* or label your own images with Roboflow / LabelImg.

Expected folder layout:
```
dataset/
  data.yaml
  train/images, train/labels
  valid/images, valid/labels
  test/images,  test/labels
```
`data.yaml` example:
```yaml
path: dataset
train: train/images
val: valid/images
test: test/images
names:
  0: pothole
  1: speed_breaker
```

In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

DATA_YAML = 'dataset/data.yaml'
RUN_TRAINING = os.path.exists(DATA_YAML)   # trains automatically once you add the dataset
print('Dataset found, training enabled' if RUN_TRAINING else f'{DATA_YAML} not found - add your dataset to enable training')

## 2.2 Train

`yolov8n` is the smallest/fastest model (good for edge devices); use `yolov8s` or `yolov8m` for higher accuracy. A GPU (e.g. Google Colab T4) makes this much faster.

In [ ]:
if RUN_TRAINING:
    model = YOLO('yolov8n.pt')
    results = model.train(data=DATA_YAML, epochs=50, imgsz=640, batch=16, patience=10, project='runs', name='road_defects')
    metrics = model.val()
    print('mAP50:', metrics.box.map50, ' mAP50-95:', metrics.box.map)

## 2.3 Run inference on an image

In [ ]:
import cv2

WEIGHTS = 'runs/road_defects/weights/best.pt'   # your trained weights

def load_model():
    return YOLO(WEIGHTS) if os.path.exists(WEIGHTS) else None

def detect_image(path, conf=0.4):
    model = load_model()
    if model is None:
        print('No trained weights yet. Train first (section 2.2).')
        return
    res = model.predict(path, conf=conf, verbose=False)[0]
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(f'{len(res.boxes)} detections')
    plt.show()
    return res

# detect_image('test_road.jpg')

## 2.4 Run on a dashcam video

Writes an annotated video and logs how many potholes / speed breakers appear in each frame.

In [ ]:
def detect_video(src, out_path='output_detected.mp4', conf=0.4, frame_skip=1):
    model = load_model()
    if model is None:
        print('No trained weights yet. Train first (section 2.2).')
        return None
    cap = cv2.VideoCapture(src)           # use 0 for a live webcam
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    w, h = int(cap.get(3)), int(cap.get(4))
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    names, log, i = model.names, [], 0
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        if i % frame_skip == 0:
            r = model.predict(frame, conf=conf, verbose=False)[0]
            frame = r.plot()
            for c in r.boxes.cls.tolist():
                log.append({'frame': i, 'time_s': i / fps, 'class': names[int(c)]})
        writer.write(frame)
        i += 1
    cap.release(); writer.release()
    print(f'Saved {out_path}; {len(log)} detections over {i} frames')
    return pd.DataFrame(log)

# log_df = detect_video('dashcam.mp4')
# log_df['class'].value_counts()

---
# Next steps

1. **Fuse both sensors**: when the camera flags a pothole and the accelerometer spikes a moment later, confirm the event and save the GPS location.
2. **Map the results**: plot confirmed events on a map with `folium`.
3. **Deploy**: export YOLO to ONNX/TFLite (`model.export(format='tflite')`) and run on a Raspberry Pi or phone.
4. **Driver alert**: trigger a warning sound when a speed breaker is detected ahead.
5. **Improve accuracy**: collect more night / rain / shadow images and real accelerometer drives at different speeds.